# D — ĐO TRẦN CỦA VIỆC CHỌN ĐOẠN (không xây gì cả)

**Câu hỏi duy nhất:** nếu bộ chấm được đọc **TẤT CẢ** các đoạn của văn bản thay vì 3 đoạn D đang
gửi, độ đúng ở **hạng 1** lên bao nhiêu?

Vì sao hỏi: kho có 34,7 đoạn/văn bản, rổ D bàn giao tối đa 3 đoạn → tầng chấm nông chỉ đọc **9%**
mỗi văn bản. Và `ce` (3 đoạn) = 0.6100 → `ce_deep` (nhiều đoạn hơn) = 0.6900: **+8 điểm chỉ vì
đọc nhiều hơn.** Đây là hiệu ứng lớn nhất đo được ngày 07/09.

## NGƯỠNG ĐẶT TRƯỚC KHI CHẠY — đọc xong không được đổi

| trần đo được | quyết định |
|---|---|
| **≥ 0,80** | chọn đoạn là cần gạt lớn nhất còn lại → D làm tiếp, ưu tiên 1 |
| 0,74 – 0,79 | có thật nhưng vừa phải → làm sau bộ phân xử cặp |
| **≤ 0,73** | đã cạn → **D DỪNG**, dồn toàn bộ GPU cho E |

Mốc so sánh: **0.7100** (`max(ce, ce_deep)`, dev300). Trần bị chặn trên bởi 0.9333 = tỉ lệ câu
có gold nằm trong top-5. Ta chỉ chấm lại top-5 nên không thể vượt con số đó.

## Cần upload lên Kaggle
1. `dev_300_locked.json`
2. `scores_dev300_fusion_M20_K20.json`
3. thư mục `selected-contexts/` (kho gốc)
4. `rerank.py` + `make_candidates_fallback.py` (dataset tiện ích, để `import` lại — **không chép code**)

GPU T4 · Internet ON. Ước ~25–40 phút.

In [ ]:
import os, sys, json, time, gc
import numpy as np, torch

# ---- SỬA 4 DÒNG NÀY CHO KHỚP TÊN DATASET CỦA BẠN ----
DEV    = "/kaggle/input/dsc-2026-dev-300-locked/dev_300_locked.json"
SCORES = "/kaggle/input/dsc-2026-scores-dev300-fusion/scores_dev300_fusion_M20_K20.json"
CTX    = "/kaggle/input/dsc-2026-dataset/DSC 2026 Dataset/selected-contexts"
UTIL   = "/kaggle/input/project-ir"          # chứa rerank.py, make_candidates_fallback.py
# ------------------------------------------------------
OUT       = "/kaggle/working/chunk_ceiling_dev300.json"
TOP_DOCS  = 5        # chỉ chấm lại top-5: trần bị chặn ở 0.9333
MAX_CHUNK = 60       # chặn văn bản dị thường; in ra số văn bản chạm trần

sys.path.insert(0, UTIL)
from rerank import load_reranker                       # tái dùng, không tự load model
from make_candidates_fallback import chunks_of, read_passage

dev = json.load(open(DEV, encoding="utf-8"))
S   = json.load(open(SCORES, encoding="utf-8"))
gold = {q: {str(x) for x in v["answer"]} for q, v in dev.items()}
Q    = list(gold)

mx    = lambda v: max(v["ce"], v.get("ce_deep", -9e9))
order = {q: [d for d, _ in sorted(S[q].items(), key=lambda kv: -mx(kv[1]))] for q in Q}
top1  = lambda pick: np.mean([pick[q] in gold[q] for q in Q])

# ---- GUARD: phải dựng lại đúng mốc 0.7100, nếu không thì nạp sai file, DỪNG ----
base = top1({q: order[q][0] for q in Q})
print(f"mốc dựng lại  = {base:.4f}   (phải là 0.7100)")
assert abs(base - 0.7100) < 1e-6, "SAI FILE ĐIỂM — dừng, đừng đọc bất kỳ Δ nào phía sau"

cap5 = np.mean([bool(gold[q] & set(order[q][:TOP_DOCS])) for q in Q])
print(f"trần lý thuyết = {cap5:.4f}   (gold nằm trong top-{TOP_DOCS})")
print(f"{len(Q)} câu · chấm lại {TOP_DOCS} văn bản/câu")

In [ ]:
# ===== Chấm TẤT CẢ đoạn của top-5 văn bản. Lưu dần, chạy lại là nối tiếp. =====
model = load_reranker("AITeamVN/Vietnamese_Reranker", device="cuda", max_length=1024)

res = json.load(open(OUT, encoding="utf-8")) if os.path.isfile(OUT) else {}
print(f"đã có {len(res)}/{len(Q)} câu, chạy tiếp phần còn lại")

_cache, capped, t0 = {}, 0, time.time()
def doc_chunks(d):
    if d not in _cache:
        c = chunks_of(read_passage(CTX, d))[:MAX_CHUNK]
        _cache[d] = c
    return _cache[d]

todo = [q for q in Q if q not in res]
for i, q in enumerate(todo, 1):
    qt = dev[q]["question"]
    pairs, owner = [], []
    for d in order[q][:TOP_DOCS]:
        ck = doc_chunks(d)
        capped += len(ck) == MAX_CHUNK
        pairs += [[qt, c] for c in ck]
        owner += [d] * len(ck)
    sc = model.predict(pairs, batch_size=32, show_progress_bar=False)
    per = {}
    for d, v in zip(owner, sc):
        per.setdefault(d, []).append(float(v))
    res[q] = per                                    # giữ TỪNG đoạn -> phân tích lại được sau

    if i % 50 == 0 or i == len(todo):
        json.dump(res, open(OUT, "w", encoding="utf-8"), ensure_ascii=False)
        el = time.time() - t0
        print(f"  {i}/{len(todo)} câu · {el/60:.1f} phút · còn ~{el/i*(len(todo)-i)/60:.1f} phút", flush=True)
        gc.collect(); torch.cuda.empty_cache()

json.dump(res, open(OUT, "w", encoding="utf-8"), ensure_ascii=False)
print(f"\nXONG. {len(res)} câu -> {OUT}   (TẢI VỀ TRƯỚC KHI ĐÓNG PHIÊN — quy tắc 2)")
print(f"văn bản chạm trần {MAX_CHUNK} đoạn: {capped}")

In [ ]:
# ===== Đọc kết quả theo đúng ngưỡng đã đặt trước =====
res = json.load(open(OUT, encoding="utf-8"))
assert len(res) == len(Q), f"mới chấm {len(res)}/{len(Q)} câu — chạy nốt ô trên rồi hãy đọc"

def pick_at(n):
    """max-pool trên n đoạn ĐIỂM CAO NHẤT của mỗi văn bản; n=None là dùng hết."""
    out = {}
    for q in Q:
        out[q] = max(res[q], key=lambda d: max(sorted(res[q][d], reverse=True)[:n or len(res[q][d])]))
    return out

print(f"mốc hiện tại (D gửi ~3 đoạn)      : {base:.4f}")
print(f"trần lý thuyết của top-{TOP_DOCS}            : {cap5:.4f}\n")
print("đọc bao nhiêu đoạn/văn bản -> đúng ở hạng 1:")
prev = None
for n in (1, 2, 3, 5, 10, 20, None):
    a = top1(pick_at(n))
    lab = "TẤT CẢ" if n is None else str(n)
    inc = "" if prev is None else f"   (+{a-prev:+.4f} so với mức trước)".replace("+-", "-")
    print(f"   {lab:>7} đoạn : {a:.4f}{inc}")
    prev = a

tran = top1(pick_at(None))
print("\n" + "=" * 62)
print(f"TRẦN CỦA VIỆC CHỌN ĐOẠN = {tran:.4f}   (mốc {base:.4f}, chênh {tran-base:+.4f})")
if   tran >= 0.80: print("=> ƯU TIÊN 1. D làm tiếp: chọn đoạn bằng bi-encoder mức đoạn.")
elif tran >= 0.74: print("=> CÓ THẬT NHƯNG VỪA PHẢI. Làm sau bộ phân xử cặp.")
else:              print("=> ĐÃ CẠN. D DỪNG hướng này, dồn GPU cho E.")
print("=" * 62)